# Entrenamiento y comparación de modelos para `no_tomo_7d`

Este notebook compara modelos de priorización de niños de 6 a 35 meses que recibieron hierro. El target es `no_tomo_7d = 1 - tomo_7d`.

La métrica primaria se mantiene igual al protocolo anterior: **PPV@20 %**, es decir, la proporción de verdaderos no consumidores dentro del 20 % con mayor riesgo. También se reportan PPV@10 %, PPV@30 %, AUC, Brier y calibración.

Decisiones principales:

- Entrenamiento/desarrollo: ENDES 2024. Evaluación temporal: ENDES 2025.
- Los conglomerados se respetan en la validación cruzada y el bootstrap.
- `atraso_cred` se calcula usando solo las medianas del conjunto de entrenamiento.
- No entran identificadores, factores muestrales, variables de toma, razones de no consumo ni variables derivadas del target.
- Se usa una sola representación por concepto para evitar redundancias. Por ejemplo, se usa `anemia_oms`, no simultáneamente hemoglobina, nivel y dos definiciones de anemia.
- `controles_cred_acum` solo sirve para construir `atraso_cred`; no entra además como predictor porque, junto con edad y atraso, sería redundante.
- Lengua materna se reserva para auditoría de equidad, no para decidir a quién priorizar.

> La evaluación temporal 2024 → 2025 es una validación de transporte entre años, pero no es totalmente externa porque parte de los conglomerados se repite. Por eso también se reporta una sensibilidad limitada a conglomerados de 2025 no vistos en 2024.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
SEMILLA = 2026
N_BOOTSTRAP = 1_000

RAIZ = Path.cwd().resolve()
if RAIZ.name.lower() == "modelo":
    RAIZ = RAIZ.parent
if not (RAIZ / "datos_procesados").exists():
    raise FileNotFoundError("Ejecuta el notebook desde la raíz del proyecto o desde modelo/.")

RUTA_DATOS = RAIZ / "datos_procesados" / "tabla_modelo_2024_2025_limpia.csv"
RUTA_RESULTADOS = RAIZ / "modelo" / "resultados_entrenamiento"
print(f"Datos: {RUTA_DATOS}")

## 1. Cohorte y target

`no_tomo_7d = 1` significa que el niño no consumió ninguna presentación de hierro durante los últimos siete días. No representa abandono futuro del tratamiento.

In [ ]:
datos = pd.read_csv(RUTA_DATOS, low_memory=False)

cohorte = datos.loc[
    datos["en_cohorte_modelo"].eq(True) & datos["tomo_7d"].isin([0, 1])
].copy()
cohorte["no_tomo_7d"] = (1 - cohorte["tomo_7d"]).astype("int8")

resumen_cohorte = (
    cohorte.groupby("anio")["no_tomo_7d"]
    .agg(n="size", no_tomo_7d="mean")
    .reset_index()
)
resumen_cohorte["no_tomo_7d"] = resumen_cohorte["no_tomo_7d"].map(lambda x: f"{x:.1%}")
display(resumen_cohorte)

assert set(cohorte["anio"].unique()) == {2024, 2025}
assert cohorte["no_tomo_7d"].isin([0, 1]).all()
assert cohorte[["anio", "caseid", "idx_nino"]].duplicated().sum() == 0

## 2. Variables y modelos

Los bloques son incrementales. Así se puede saber si las variables nuevas realmente mejoran sobre edad y sobre el modelo operativo anterior. `M2` conserva las ideas del compañero —edad, atraso de controles y presentación— pero elimina la redundancia de incluir simultáneamente controles crudos y atraso.

In [ ]:
PRESENTACIONES = ["recibio_jarabe", "recibio_gotas", "recibio_polvo"]

VARIABLES_M1 = ["edad_meses"]
VARIABLES_M2 = ["edad_meses", "atraso_cred", *PRESENTACIONES]
VARIABLES_M3 = [*VARIABLES_M2, "n_cred_con_fecha", "lugar_cred_codigo"]
VARIABLES_M4 = [
    *VARIABLES_M3,
    "anemia_oms",
    "edad_madre",
    "nivel_educativo_madre",
    "cuidador_habitual",
    "quintil_riqueza",
    "n_menores_5_hogar",
    "personas_por_dormitorio",
    "area",
]

VARIABLES_POR_MODELO = {
    "M1_edad": VARIABLES_M1,
    "M2_operativo": VARIABLES_M2,
    "M3_CRED": VARIABLES_M3,
    "M4_ampliado": VARIABLES_M4,
    "M5_bosque": VARIABLES_M4,
}

NUMERICAS = {
    "atraso_cred", "n_cred_con_fecha", "edad_madre",
    "n_menores_5_hogar", "personas_por_dormitorio"
}
BINARIAS = set(PRESENTACIONES)
CATEGORICAS = {
    "lugar_cred_codigo", "anemia_oms", "nivel_educativo_madre",
    "cuidador_habitual", "quintil_riqueza", "area"
}

descripcion_variables = pd.DataFrame([
    ("edad_meses", "niño", "Principal patrón por edad; se modela con splines en regresión logística."),
    ("atraso_cred", "CRED derivada", "Controles acumulados menos la mediana para su edad, calculada solo en entrenamiento."),
    ("recibio_jarabe/gotas/polvo", "tratamiento", "Presentación recibida; mantiene las variables del modelo previo."),
    ("n_cred_con_fecha", "CRED", "Cantidad de controles respaldados por fecha en el carné."),
    ("lugar_cred_codigo", "acceso", "Tipo de establecimiento donde se realizó el CRED."),
    ("anemia_oms", "clínica", "Una sola definición clínica; no se añaden Hb, nivel y MINSA simultáneamente."),
    ("edad_madre", "madre", "Etapa de vida de la cuidadora."),
    ("nivel_educativo_madre", "madre", "Contexto educativo; se trata como categoría."),
    ("cuidador_habitual", "cuidados", "Quién cuida al niño cuando la madre se ausenta."),
    ("quintil_riqueza", "hogar", "Contexto socioeconómico; se trata como categoría."),
    ("n_menores_5_hogar", "hogar", "Carga de cuidado de niños pequeños."),
    ("personas_por_dormitorio", "hogar", "Hacinamiento sin repetir residentes y habitaciones por separado."),
    ("area", "acceso", "Contexto urbano/rural; obliga a revisar equidad por área."),
], columns=["variable", "bloque", "motivo"])
display(descripcion_variables)

In [ ]:
# Controles explícitos contra fuga de información y duplicación conceptual.
PROHIBIDAS = {
    "no_tomo_7d", "tomo_7d", "tomo7_jarabe", "tomo7_gotas",
    "tomo7_polvo", "tomo7_otra", "tomo_estado",
    "razon_no_consumo_cod", "razon_estomago", "razon_diarrea",
    "razon_estrenimiento", "razon_sabor", "razon_dientes",
    "razon_orientacion", "razon_inicio_suplementacion",
    "razon_olvido", "razon_otra",
}
IDENTIFICADORES_DISENO = {
    "anio", "caseid", "idx_nino", "hhid", "conglomerado",
    "estrato", "factor_expansion_crudo", "factor_hogar_crudo"
}
todas_predictoras = set().union(*map(set, VARIABLES_POR_MODELO.values()))

assert not (todas_predictoras & PROHIBIDAS), todas_predictoras & PROHIBIDAS
assert not (todas_predictoras & IDENTIFICADORES_DISENO), todas_predictoras & IDENTIFICADORES_DISENO
assert "controles_cred_acum" not in todas_predictoras
assert "atraso_cred" in todas_predictoras
assert not ({"anemia_minsa", "nivel_anemia_oms", "nivel_anemia_minsa",
             "hb_ajustada_minsa_gdl", "hb_ajustada_oms_gdl"} & todas_predictoras)
assert len(VARIABLES_M4) == len(set(VARIABLES_M4))
print(f"Predictores únicos en el modelo ampliado: {len(VARIABLES_M4)}")

## 3. Funciones de preparación, métricas y modelos

La función PPV resuelve empates usando el valor esperado dentro del grupo empatado. Esto evita que el resultado dependa del orden accidental de las filas, especialmente en el modelo que usa solo edad.

In [ ]:
def agregar_atraso_cred(entrenamiento, evaluacion):
    entrenamiento = entrenamiento.copy()
    evaluacion = evaluacion.copy()
    mediana_por_edad = entrenamiento.groupby("edad_meses")["controles_cred_acum"].median()
    mediana_global = entrenamiento["controles_cred_acum"].median()

    for tabla in (entrenamiento, evaluacion):
        referencia = tabla["edad_meses"].map(mediana_por_edad).fillna(mediana_global)
        tabla["atraso_cred"] = tabla["controles_cred_acum"] - referencia
    return entrenamiento, evaluacion, mediana_por_edad


def ppv_en_fraccion(y, puntaje, fraccion):
    y = np.asarray(y, dtype=float)
    puntaje = np.asarray(puntaje, dtype=float)
    n_elegir = max(1, int(np.ceil(fraccion * len(y))))
    umbral = np.partition(puntaje, len(puntaje) - n_elegir)[len(puntaje) - n_elegir]
    superiores = puntaje > umbral
    empatados = puntaje == umbral
    faltan = n_elegir - superiores.sum()
    positivos_esperados = y[superiores].sum()
    if faltan > 0:
        positivos_esperados += faltan * y[empatados].mean()
    return float(positivos_esperados / n_elegir)


def calibracion(y, puntaje):
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(puntaje, dtype=float), 1e-6, 1 - 1e-6)
    if np.std(p) < 1e-12 or len(np.unique(y)) < 2:
        return np.nan, np.nan
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    ajuste = LogisticRegression(C=1e6, max_iter=2_000).fit(logit, y)
    return float(ajuste.intercept_[0]), float(ajuste.coef_[0, 0])


def metricas(y, puntaje):
    y = np.asarray(y, dtype=int)
    puntaje = np.asarray(puntaje, dtype=float)
    intercepto, pendiente = calibracion(y, puntaje)
    return {
        "n": len(y),
        "prevalencia": y.mean(),
        "ppv10": ppv_en_fraccion(y, puntaje, 0.10),
        "ppv20": ppv_en_fraccion(y, puntaje, 0.20),
        "ppv30": ppv_en_fraccion(y, puntaje, 0.30),
        "auc": roc_auc_score(y, puntaje) if len(np.unique(y)) == 2 else np.nan,
        "brier": brier_score_loss(y, puntaje),
        "prediccion_media": puntaje.mean(),
        "calibracion_intercepto": intercepto,
        "calibracion_pendiente": pendiente,
    }


def crear_preprocesador(variables):
    variables = list(variables)
    transformadores = []
    if "edad_meses" in variables:
        transformadores.append((
            "edad",
            Pipeline([
                ("imputar", SimpleImputer(strategy="median")),
                ("spline", SplineTransformer(n_knots=5, degree=3, include_bias=False)),
            ]),
            ["edad_meses"],
        ))
    numericas = [v for v in variables if v in NUMERICAS]
    binarias = [v for v in variables if v in BINARIAS]
    categoricas = [v for v in variables if v in CATEGORICAS]
    if numericas:
        transformadores.append((
            "num",
            Pipeline([
                ("imputar", SimpleImputer(strategy="median", add_indicator=True)),
                ("escalar", StandardScaler()),
            ]),
            numericas,
        ))
    if binarias:
        transformadores.append((
            "bin", SimpleImputer(strategy="most_frequent", add_indicator=True), binarias
        ))
    if categoricas:
        transformadores.append((
            "cat",
            Pipeline([
                ("imputar", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                ("onehot", OneHotEncoder(
                    handle_unknown="infrequent_if_exist", min_frequency=20
                )),
            ]),
            categoricas,
        ))
    return ColumnTransformer(transformadores, remainder="drop")


def crear_modelo(nombre):
    variables = VARIABLES_POR_MODELO[nombre]
    pre = crear_preprocesador(variables)
    if nombre == "M5_bosque":
        estimador = RandomForestClassifier(
            n_estimators=300, min_samples_leaf=20, max_features="sqrt",
            n_jobs=-1, random_state=SEMILLA
        )
    else:
        estimador = LogisticRegression(max_iter=3_000)
    return Pipeline([("pre", pre), ("modelo", estimador)])

## 4. Desarrollo: validación cruzada agrupada en 2024

La elección del candidato se hace únicamente con 2024. Cada conglomerado aparece completo en entrenamiento o validación, nunca en ambos. `M0` es la prevalencia del fold de entrenamiento y representa una priorización sin información individual.

In [ ]:
desarrollo = cohorte.loc[cohorte["anio"] == 2024].reset_index(drop=True)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
filas_cv = []

for fold, (i_entrena, i_valida) in enumerate(
    cv.split(desarrollo, desarrollo["no_tomo_7d"], groups=desarrollo["conglomerado"]),
    start=1,
):
    entrena = desarrollo.iloc[i_entrena].copy()
    valida = desarrollo.iloc[i_valida].copy()
    entrena, valida, _ = agregar_atraso_cred(entrena, valida)
    y_entrena = entrena["no_tomo_7d"].to_numpy()
    y_valida = valida["no_tomo_7d"].to_numpy()

    p_azar = np.full(len(valida), y_entrena.mean())
    filas_cv.append({"fold": fold, "modelo": "M0_azar", **metricas(y_valida, p_azar)})

    for nombre in VARIABLES_POR_MODELO:
        modelo = crear_modelo(nombre)
        variables = VARIABLES_POR_MODELO[nombre]
        modelo.fit(entrena[variables], y_entrena)
        p = modelo.predict_proba(valida[variables])[:, 1]
        filas_cv.append({"fold": fold, "modelo": nombre, **metricas(y_valida, p)})

resultados_cv_folds = pd.DataFrame(filas_cv)
columnas_resumen = ["ppv10", "ppv20", "ppv30", "auc", "brier"]
resultados_cv = resultados_cv_folds.groupby("modelo")[columnas_resumen].agg(["mean", "std"])
resultados_cv.columns = [f"{metrica}_{estadistico}" for metrica, estadistico in resultados_cv.columns]
resultados_cv = resultados_cv.sort_values(["ppv20_mean", "auc_mean"], ascending=False)
display(resultados_cv.style.format("{:.3f}"))

In [ ]:
# Se elige el mejor candidato ampliado solo con CV 2024; M0 y M1 son referencias.
candidatos = ["M2_operativo", "M3_CRED", "M4_ampliado", "M5_bosque"]
candidato = (
    resultados_cv.loc[candidatos]
    .sort_values(["ppv20_mean", "auc_mean"], ascending=False)
    .index[0]
)
print(f"Candidato elegido usando solo 2024: {candidato}")
print("La elección final debe considerar también simplicidad, calibración, equidad y disponibilidad real de variables.")

## 5. Evaluación temporal 2024 → 2025

Después de elegir el candidato con 2024, se reajusta cada modelo con todo 2024 y se evalúa una sola vez en 2025. El target de 2025 nunca se usa para imputar, transformar ni construir `atraso_cred`.

In [ ]:
entrena_2024 = cohorte.loc[cohorte["anio"] == 2024].reset_index(drop=True)
prueba_2025 = cohorte.loc[cohorte["anio"] == 2025].reset_index(drop=True)
entrena_2024, prueba_2025, mediana_cred_2024 = agregar_atraso_cred(entrena_2024, prueba_2025)

y_entrena = entrena_2024["no_tomo_7d"].to_numpy()
y_prueba = prueba_2025["no_tomo_7d"].to_numpy()
predicciones_2025 = {"M0_azar": np.full(len(prueba_2025), y_entrena.mean())}
modelos_ajustados = {}

for nombre, variables in VARIABLES_POR_MODELO.items():
    modelo = crear_modelo(nombre)
    modelo.fit(entrena_2024[variables], y_entrena)
    modelos_ajustados[nombre] = modelo
    predicciones_2025[nombre] = modelo.predict_proba(prueba_2025[variables])[:, 1]

resultados_temporales = pd.DataFrame([
    {"modelo": nombre, **metricas(y_prueba, p)}
    for nombre, p in predicciones_2025.items()
]).set_index("modelo").sort_values(["ppv20", "auc"], ascending=False)

columnas_mostrar = [
    "n", "prevalencia", "ppv10", "ppv20", "ppv30", "auc",
    "brier", "prediccion_media", "calibracion_intercepto",
    "calibracion_pendiente"
]
display(resultados_temporales[columnas_mostrar].style.format({
    "prevalencia": "{:.1%}", "ppv10": "{:.1%}",
    "ppv20": "{:.1%}", "ppv30": "{:.1%}",
    "auc": "{:.3f}", "brier": "{:.3f}",
    "prediccion_media": "{:.1%}",
    "calibracion_intercepto": "{:.3f}",
    "calibracion_pendiente": "{:.3f}",
}))

## 6. Incertidumbre de la mejora frente a usar solo edad

Se remuestrean conglomerados completos de 2025. La comparación primaria es la diferencia de PPV@20 % entre el candidato elegido en CV 2024 y `M1_edad`.

In [ ]:
rng = np.random.default_rng(SEMILLA)
grupos = prueba_2025["conglomerado"].to_numpy()
grupos_unicos = np.unique(grupos)
indices_grupo = {g: np.flatnonzero(grupos == g) for g in grupos_unicos}
diferencias_ppv20 = []
diferencias_auc = []

for _ in range(N_BOOTSTRAP):
    muestra_grupos = rng.choice(grupos_unicos, size=len(grupos_unicos), replace=True)
    indices = np.concatenate([indices_grupo[g] for g in muestra_grupos])
    y_b = y_prueba[indices]
    p_candidato = predicciones_2025[candidato][indices]
    p_edad = predicciones_2025["M1_edad"][indices]
    diferencias_ppv20.append(
        ppv_en_fraccion(y_b, p_candidato, 0.20) - ppv_en_fraccion(y_b, p_edad, 0.20)
    )
    if len(np.unique(y_b)) == 2:
        diferencias_auc.append(roc_auc_score(y_b, p_candidato) - roc_auc_score(y_b, p_edad))

dif_ppv = resultados_temporales.loc[candidato, "ppv20"] - resultados_temporales.loc["M1_edad", "ppv20"]
dif_auc = resultados_temporales.loc[candidato, "auc"] - resultados_temporales.loc["M1_edad", "auc"]
resumen_bootstrap = pd.DataFrame({
    "metrica": ["Diferencia PPV@20", "Diferencia AUC"],
    "estimacion": [dif_ppv, dif_auc],
    "ic95_inferior": [np.percentile(diferencias_ppv20, 2.5), np.percentile(diferencias_auc, 2.5)],
    "ic95_superior": [np.percentile(diferencias_ppv20, 97.5), np.percentile(diferencias_auc, 97.5)],
})
display(resumen_bootstrap.style.format({
    "estimacion": "{:+.3f}", "ic95_inferior": "{:+.3f}", "ic95_superior": "{:+.3f}"
}))

## 7. Sensibilidad: solo conglomerados de 2025 no observados en 2024

In [ ]:
conglomerados_2024 = set(entrena_2024["conglomerado"].unique())
no_vistos = ~prueba_2025["conglomerado"].isin(conglomerados_2024)
y_no_vistos = y_prueba[no_vistos.to_numpy()]

resultados_no_vistos = pd.DataFrame([
    {"modelo": nombre, **metricas(y_no_vistos, p[no_vistos.to_numpy()])}
    for nombre, p in predicciones_2025.items()
]).set_index("modelo").sort_values(["ppv20", "auc"], ascending=False)

print(f"Niños 2025 en conglomerados no vistos en 2024: {no_vistos.sum():,} de {len(prueba_2025):,} ({no_vistos.mean():.1%})")
display(resultados_no_vistos[["n", "prevalencia", "ppv20", "auc", "brier"]].style.format({
    "prevalencia": "{:.1%}", "ppv20": "{:.1%}",
    "auc": "{:.3f}", "brier": "{:.3f}"
}))

## 8. Equidad del candidato en 2025

Se auditan área, educación, riqueza y código de lengua materna. La lengua no entra al modelo. Los grupos con menos de 100 observaciones no se reportan porque sus métricas serían muy inestables.

In [ ]:
auditoria = prueba_2025.copy()
auditoria["puntaje"] = predicciones_2025[candidato]
filas_equidad = []

for variable in ["area", "nivel_educativo_madre", "quintil_riqueza", "lengua_materna"]:
    for grupo, sub in auditoria.groupby(variable, dropna=False):
        if len(sub) < 100:
            continue
        y_g = sub["no_tomo_7d"].to_numpy()
        p_g = sub["puntaje"].to_numpy()
        filas_equidad.append({
            "variable": variable, "grupo": str(grupo), "n": len(sub),
            "observado": y_g.mean(), "predicho": p_g.mean(),
            "ppv20": ppv_en_fraccion(y_g, p_g, 0.20),
            "auc": roc_auc_score(y_g, p_g) if len(np.unique(y_g)) == 2 else np.nan,
        })

resultados_equidad = pd.DataFrame(filas_equidad)
display(resultados_equidad.style.format({
    "observado": "{:.1%}", "predicho": "{:.1%}",
    "ppv20": "{:.1%}", "auc": "{:.3f}"
}))

## 9. Variables transformadas con mayor peso

Los coeficientes o importancias ayudan a auditar el modelo, pero no demuestran causalidad. En variables categóricas representan categorías transformadas y en edad representan bases spline.

In [ ]:
modelo_candidato = modelos_ajustados[candidato]
nombres_transformados = modelo_candidato.named_steps["pre"].get_feature_names_out()
estimador = modelo_candidato.named_steps["modelo"]

if hasattr(estimador, "coef_"):
    valores = estimador.coef_[0]
    tipo_valor = "coeficiente"
else:
    valores = estimador.feature_importances_
    tipo_valor = "importancia"

importancias = pd.DataFrame({
    "variable_transformada": nombres_transformados,
    tipo_valor: valores,
})
importancias["magnitud"] = importancias[tipo_valor].abs()
importancias = importancias.sort_values("magnitud", ascending=False).drop(columns="magnitud")
display(importancias.head(25))

## 10. Guardar reportes

Estos archivos contienen resultados y auditorías, no el modelo de producción. El modelo solo debe exportarse después de revisar desempeño, calibración, equidad y disponibilidad de variables en la API.

In [ ]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
resultados_cv_folds.to_csv(RUTA_RESULTADOS / "validacion_cruzada_2024_folds.csv", index=False)
resultados_cv.reset_index().to_csv(RUTA_RESULTADOS / "validacion_cruzada_2024_resumen.csv", index=False)
resultados_temporales.reset_index().to_csv(RUTA_RESULTADOS / "evaluacion_temporal_2025.csv", index=False)
resultados_no_vistos.reset_index().to_csv(RUTA_RESULTADOS / "evaluacion_2025_conglomerados_no_vistos.csv", index=False)
resumen_bootstrap.to_csv(RUTA_RESULTADOS / "bootstrap_candidato_vs_edad.csv", index=False)
resultados_equidad.to_csv(RUTA_RESULTADOS / "equidad_candidato_2025.csv", index=False)
descripcion_variables.to_csv(RUTA_RESULTADOS / "variables_seleccionadas.csv", index=False)
importancias.to_csv(RUTA_RESULTADOS / "importancias_candidato.csv", index=False)

print(f"Candidato seleccionado en CV 2024: {candidato}")
print(f"Reportes guardados en: {RUTA_RESULTADOS}")

## Criterio de lectura

1. Primero revisar la media y variabilidad de PPV@20 en CV 2024.
2. Después verificar si la mejora temporal frente a `M1_edad` tiene un IC 95 % que no incluya cero.
3. Confirmar que la mejora se mantiene en conglomerados no vistos.
4. Revisar Brier, intercepto y pendiente de calibración; una mejor AUC no garantiza probabilidades correctas.
5. Revisar diferencias por área, educación, riqueza y lengua.
6. Si el modelo ampliado no mejora de forma clara, conservar el modelo más simple.
7. Antes de producción, verificar que todas las variables elegidas existen en el formulario o API y definir una política de datos faltantes.